<a href="https://colab.research.google.com/github/UlaStats/MSc-project-pipe-failure-prediction/blob/main/Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MSc Project - Modelling

This notebook contains code for MSc project on modelling time-to-failure of water mains. Machine learning regression models are trained - XGBoost, Random Forests, SVR and ANN. Deep learning models are also trained - RNN and LSTM.

# Import data and required packages

In [2]:
# import packages

import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import max_error
import matplotlib.pyplot as plt
from google.colab import drive
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import statistics


In [21]:
# mount Google Drive

drive.mount("/content/drive", force_remount = True)

Mounted at /content/drive


In [4]:
# import data

attributes = pd.read_csv("/content/drive/MyDrive/MSc project/attributes.csv", encoding = "latin1")

X_train = pd.read_csv("/content/drive/MyDrive/MSc project/X_train.csv", encoding = "latin1")
X_valid = pd.read_csv("/content/drive/MyDrive/MSc project/X_valid.csv", encoding = "latin1")
X_test = pd.read_csv("/content/drive/MyDrive/MSc project/X_test.csv", encoding = "latin1")


y_train = pd.read_csv("/content/drive/MyDrive/MSc project/y_train.csv", encoding = "latin1")
y_valid = pd.read_csv("/content/drive/MyDrive/MSc project/y_valid.csv", encoding = "latin1")
y_test = pd.read_csv("/content/drive/MyDrive/MSc project/y_test.csv", encoding = "latin1")

In [5]:
# convert categorical columns

X_train['Lining'] = X_train['Lining'].astype("category")
X_train['Surface'] = X_train['Surface'].astype("category")
X_train['Soil'] = X_train['Soil'].astype("category")
X_train['Material'] = X_train['Material'].astype("category")


X_valid['Lining'] = X_valid['Lining'].astype("category")
X_valid['Surface'] = X_valid['Surface'].astype("category")
X_valid['Soil'] = X_valid['Soil'].astype("category")
X_valid['Material'] = X_valid['Material'].astype("category")


X_test['Lining'] = X_test['Lining'].astype("category")
X_test['Surface'] = X_test['Surface'].astype("category")
X_test['Soil'] = X_test['Soil'].astype("category")
X_test['Material'] = X_test['Material'].astype("category")


In [6]:
# convert to tidy data

X_train_tidy = pd.get_dummies(X_train)
X_test_tidy = pd.get_dummies(X_test)

In [7]:
# normalise data

scaler = StandardScaler()


X_train_tidy['Diameter'] = scaler.fit_transform(X_train_tidy[['Diameter']])
X_train_tidy['Length'] = scaler.fit_transform(X_train_tidy[['Length']])
X_train_tidy['Soil_pH'] = scaler.fit_transform(X_train_tidy[['Soil_pH']])
X_train_tidy['Frost_days'] = scaler.fit_transform(X_train_tidy[['Frost_days']])
X_train_tidy['Hydrogen Ion'] = scaler.fit_transform(X_train_tidy[['Hydrogen Ion']])
X_train_tidy['Free chlorine'] = scaler.fit_transform(X_train_tidy[['Free chlorine']])
X_train_tidy['Age'] = scaler.fit_transform(X_train_tidy[['Age']])
X_train_tidy['Previous bursts'] = scaler.fit_transform(X_train_tidy[["Previous bursts"]])

X_test_tidy['Diameter'] = scaler.fit_transform(X_test_tidy[['Diameter']])
X_test_tidy['Length'] = scaler.fit_transform(X_test_tidy[['Length']])
X_test_tidy['Soil_pH'] = scaler.fit_transform(X_test_tidy[['Soil_pH']])
X_test_tidy['Frost_days'] = scaler.fit_transform(X_test_tidy[['Frost_days']])
X_test_tidy['Hydrogen Ion'] = scaler.fit_transform(X_test_tidy[['Hydrogen Ion']])
X_test_tidy['Free chlorine'] = scaler.fit_transform(X_test_tidy[['Free chlorine']])
X_test_tidy['Age'] = scaler.fit_transform(X_test_tidy[['Age']])
X_test_tidy['Previous bursts'] = scaler.fit_transform(X_test_tidy[["Previous bursts"]])

# Modelling

In thi section, model training is done for XGBoost, ANN,

### XGBoost

In [8]:
# converting dataframe to DMatrix

import xgboost as xgb

xgb_train_tidy = xgb.DMatrix(X_train_tidy, y_train, enable_categorical=True)
xgb_valid = xgb.DMatrix(X_valid, y_valid, enable_categorical=True)
xgb_test_tidy = xgb.DMatrix(X_test_tidy, y_test, enable_categorical=True)




In [9]:
# model training of XGBoost

params = {
    'objective': "reg:squarederror"
}


model_xgb = xgb.train(params = params, dtrain = xgb_train_tidy)

In [11]:
# max error for XGBoost

preds = model_xgb.predict(xgb_test_tidy)

print(max_error(y_test, preds))

In [14]:
# MAE

MAE = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    xgb_test = xgb.DMatrix(X_test_bootstrap, y_test_bootrstrap, enable_categorical=True)

    preds = model_xgb.predict(xgb_test)

    MAE_bootstrap = mean_absolute_error(y_test_bootrstrap, preds)

    MAE.append(MAE_bootstrap)




In [25]:
RMSE = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    xgb_test = xgb.DMatrix(X_test_bootstrap, y_test_bootrstrap, enable_categorical=True)

    preds = model_xgb.predict(xgb_test)

    RMSE_bootstrap = root_mean_squared_error(y_test_bootrstrap, preds)

    RMSE.append(RMSE_bootstrap)




In [28]:
R2 = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    xgb_test = xgb.DMatrix(X_test_bootstrap, y_test_bootrstrap, enable_categorical=True)

    preds = model_xgb.predict(xgb_test)

    R2_bootstrap = r2_score(y_test_bootrstrap, preds)

    R2.append(R2_bootstrap)


In [42]:
statistics.mean(MAE)

4.580506386756897

In [27]:
statistics.stdev(MAE)

0.07654833806602741

In [43]:
statistics.mean(RMSE)

5.617972005367279

In [26]:
statistics.stdev(RMSE)

0.08310041789721537

In [44]:
statistics.mean(R2)

0.2224479779601097

In [30]:
statistics.stdev(R2)

0.018948345195753794

In [80]:
MAE_XGBoost = pd.DataFrame(MAE)
MAE_XGBoost.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/MAE_XGBoost.csv", encoding ='latin1', index=False)

In [81]:
RMSE_XGBoost = pd.DataFrame(RMSE)
RMSE_XGBoost.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/RMSE_XGBoost.csv", encoding ='latin1', index=False)

In [82]:
R2_XGBoost = pd.DataFrame(R2)
R2_XGBoost.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/R2_XGBoost.csv", encoding ='latin1', index=False)

### Random Forest

In [31]:
from sklearn.ensemble import RandomForestRegressor

model_RF = RandomForestRegressor()
model_RF.fit(X_train_tidy, y_train)
y_pred_RF = model_RF.predict(X_test_tidy)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [32]:
# max error for RF

preds = model_RF.predict(X_test_tidy)

print(max_error(y_test, preds))

15.876136986301379


In [33]:
MAE_RF = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_RF.predict(X_test_bootstrap)

    MAE_bootstrap = mean_absolute_error(y_test_bootrstrap, preds)

    MAE_RF.append(MAE_bootstrap)


In [40]:
statistics.mean(MAE_RF)

4.626234525319556

In [34]:
statistics.stdev(MAE_RF)

0.0771686643561807

In [39]:
RMSE_RF = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_RF.predict(X_test_bootstrap)

    RMSE_bootstrap = root_mean_squared_error(y_test_bootrstrap, preds)

    RMSE_RF.append(RMSE_bootstrap)


KeyboardInterrupt: 

In [41]:
statistics.mean(RMSE_RF)

5.673279735360421

In [37]:
statistics.stdev(RMSE_RF)

0.08634372189033054

In [45]:
R2_RF = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_RF.predict(X_test_bootstrap)

    R2_bootstrap = r2_score(y_test_bootrstrap, preds)

    R2_RF.append(R2_bootstrap)


In [46]:
statistics.mean(R2_RF)

0.20775400632872026

In [47]:
statistics.stdev(R2_RF)

0.01970435898044576

In [77]:
MAE_RF = pd.DataFrame(MAE_RF)
MAE_RF.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/MAE_RF.csv", encoding ='latin1', index=False)

In [78]:
RMSE_RF = pd.DataFrame(RMSE_RF)
RMSE_RF.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/RMSE_RF.csv", encoding ='latin1', index=False)

In [79]:
R2_RF = pd.DataFrame(R2_RF)
R2_RF.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/R2_RF.csv", encoding ='latin1', index=False)

### Support Vector Regression

In [60]:
from sklearn.svm import SVR

model_SVR = SVR()
model_SVR.fit(X_train_tidy, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


SVR()

In [83]:
# max error for SVR


preds = model_SVR.predict(X_test_tidy)

print(max_error(y_test, preds))

15.882773889356402


In [61]:
MAE_SVR = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_SVR.predict(X_test_bootstrap)

    MAE_bootstrap = mean_absolute_error(y_test_bootrstrap, preds)

    MAE_SVR.append(MAE_bootstrap)

In [64]:


statistics.mean(MAE_SVR)

4.706420649855823

In [65]:
statistics.stdev(MAE_SVR)

0.08187276400889291

In [62]:
RMSE_SVR = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_SVR.predict(X_test_bootstrap)

    RMSE_bootstrap = root_mean_squared_error(y_test_bootrstrap, preds)

    RMSE_SVR.append(RMSE_bootstrap)

In [66]:
statistics.mean(RMSE_SVR)

5.894148645937151

In [67]:
statistics.stdev(RMSE_SVR)

0.09326032186733413

In [63]:
R2_SVR = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_SVR.predict(X_test_bootstrap)

    R2_bootstrap = r2_score(y_test_bootrstrap, preds)

    R2_SVR.append(R2_bootstrap)

In [68]:
statistics.mean(R2_SVR)

0.1452882064738485

In [69]:
statistics.stdev(R2_SVR)

0.01538112320588628

In [70]:
MAE_SVR = pd.DataFrame(MAE_SVR)
MAE_SVR.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/MAE_SVR.csv", encoding ='latin1', index=False)

In [72]:
RMSE_SVR = pd.DataFrame(RMSE_SVR)
RMSE_SVR.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/RMSE_SVR.csv", encoding ='latin1', index=False)

In [73]:
R2_SVR = pd.DataFrame(R2_SVR)
R2_SVR.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/R2_SVR.csv", encoding ='latin1', index=False)

### ANN

In [48]:
from sklearn.neural_network import MLPRegressor

In [49]:
model_ANN = MLPRegressor()

In [50]:
model_ANN.fit(X_train_tidy,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:1650: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor()

In [85]:
# max error for ANN


preds = model_ANN.predict(X_test_tidy)

print(max_error(y_test, preds))

16.657239246173685


In [51]:
# bootstrapping for MAE

MAE_ANN = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_ANN.predict(X_test_bootstrap)

    MAE_bootstrap = mean_absolute_error(y_test_bootrstrap, preds)

    MAE_ANN.append(MAE_bootstrap)

In [52]:
statistics.mean(MAE_ANN)

4.596823571124361

In [53]:
statistics.stdev(MAE_ANN)

0.0809824436524317

In [54]:
# bootstrapping for RMSE

RMSE_ANN = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_ANN.predict(X_test_bootstrap)

    RMSE_bootstrap = root_mean_squared_error(y_test_bootrstrap, preds)

    RMSE_ANN.append(RMSE_bootstrap)

In [55]:
statistics.mean(RMSE_ANN)

5.6822974161067075

In [56]:
statistics.stdev(RMSE_ANN)

0.08280794464197869

In [57]:
R2_ANN = []

for i in range(1000):

    X_test_bootstrap = X_test_tidy.sample(n = 1734, replace = True)

    y_test_bootrstrap = y_test.iloc[X_test_bootstrap.index]

    preds = model_ANN.predict(X_test_bootstrap)

    R2_bootstrap = r2_score(y_test_bootrstrap, preds)

    R2_ANN.append(R2_bootstrap)

In [58]:
statistics.mean(R2_ANN)

0.2069341817367622

In [59]:
statistics.stdev(R2_Ann)

0.019974893669718247

In [74]:
MAE_ANN = pd.DataFrame(MAE_ANN)
MAE_ANN.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/MAE_ANN.csv", encoding ='latin1', index=False)

In [75]:
RMSE_ANN = pd.DataFrame(RMSE_ANN)
RMSE_ANN.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/RMSE_ANN.csv", encoding ='latin1', index=False)

In [76]:
R2_ANN = pd.DataFrame(R2_ANN)
R2_ANN.to_csv("/content/drive/MyDrive/MSc project/Experiment 1/R2_ANN.csv", encoding ='latin1', index=False)